# Construction d'une table de mortalité d'expérience

## Objectif
Ce notebook construit une table de mortalité d'expérience à partir d'un portefeuille assuré, en suivant les étapes suivantes :
1. Chargement et contrôle qualité des données
2. Calcul des expositions et taux bruts de mortalité
3. Lissage des taux (Whittaker-Henderson)
4. Comparaison avec une table de référence (TH/TF 00-02)
5. Calcul du SMR (Standardised Mortality Ratio)
6. Export de la table finale

## Hypothèses
- Les données d'entrée sont au format **une ligne par contrat** avec les colonnes décrites ci-dessous
- Les expositions sont calculées en **années-personnes** (méthode Poisson)
- La table de référence utilisée est la **TH 00-02** (hommes) ou **TF 00-02** (femmes)
- Le lissage est appliqué sur la plage d'âges **[20, 90]** — en dehors, on extrapole

## Structure attendue des données d'entrée (`data`)
La variable `data` doit être un `pd.DataFrame` avec au minimum :

| Colonne | Type | Description |
|---|---|---|
| `id` | str | Identifiant unique du contrat |
| `date_naissance` | date | Date de naissance de l'assuré |
| `date_entree` | date | Date d'entrée en observation |
| `date_sortie` | date | Date de sortie (résiliation, décès, fin d'étude) |
| `cause_sortie` | str | `'deces'` ou `'autre'` |
| `sexe` | str | `'H'` ou `'F'` |

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded
import warnings
warnings.filterwarnings('ignore')

print('Librairies chargées.')

## Étape 1 — Chargement des données

Charger ici le fichier source et l'affecter à la variable `data`.
La source peut être un CSV, une base de données, ou une construction manuelle.
À l'issue de cette cellule, `data` doit respecter la structure décrite en introduction.

In [ ]:
# Exemple : chargement depuis un CSV
# Adapter le chemin selon l'environnement
# data = pd.read_csv('path/to/portefeuille.csv', parse_dates=['date_naissance','date_entree','date_sortie'])

# Exemple de données synthétiques pour test
np.random.seed(42)
n = 5000
date_ref = pd.Timestamp('2020-01-01')
ages = np.random.randint(30, 85, n)
data = pd.DataFrame({
    'id': [f'C{i:05d}' for i in range(n)],
    'date_naissance': [date_ref - pd.DateOffset(years=int(a), days=np.random.randint(0,365)) for a in ages],
    'date_entree': [date_ref - pd.DateOffset(days=np.random.randint(365, 1825)) for _ in range(n)],
    'date_sortie': [date_ref + pd.DateOffset(days=np.random.randint(0, 1460)) for _ in range(n)],
    'cause_sortie': np.random.choice(['deces','autre'], n, p=[0.03, 0.97]),
    'sexe': np.random.choice(['H','F'], n)
})

print(f'Données chargées : {len(data):,} contrats')
print(data.head())

## Étape 2 — Contrôles qualité

Vérifier :
- Absence de valeurs manquantes sur les colonnes clés
- Cohérence des dates (date_entree < date_sortie, date_naissance < date_entree)
- Distribution des âges et des causes de sortie

Les lignes non conformes sont **supprimées** avec un log.

In [ ]:
n_init = len(data)

# Colonnes obligatoires
required_cols = ['id','date_naissance','date_entree','date_sortie','cause_sortie','sexe']
assert all(c in data.columns for c in required_cols), f'Colonnes manquantes : {set(required_cols)-set(data.columns)}'

# Parse dates si nécessaire
for col in ['date_naissance','date_entree','date_sortie']:
    data[col] = pd.to_datetime(data[col])

# Suppression des incohérences
mask_ok = (
    data['date_entree'] < data['date_sortie']
) & (
    data['date_naissance'] < data['date_entree']
) & (
    data['cause_sortie'].isin(['deces','autre'])
) & (
    data['sexe'].isin(['H','F'])
)
data = data[mask_ok].copy()

print(f'Lignes supprimées (contrôle qualité) : {n_init - len(data)}')
print(f'Lignes conservées : {len(data):,}')
print(f"\nRépartition sexe :\n{data['sexe'].value_counts()}")
print(f"\nCauses de sortie :\n{data['cause_sortie'].value_counts()}")

## Étape 3 — Calcul des expositions et taux bruts

Pour chaque âge entier `x`, on calcule :
- **E_x** : exposition centrale (années-personnes passées à l'âge x)
- **D_x** : nombre de décès à l'âge x
- **mu_x** = D_x / E_x : taux central de mortalité brut
- **q_x** approché : 1 - exp(-mu_x)

L'âge est calculé à la date d'entrée en observation (âge en années révolues).

In [ ]:
# Calcul de l'âge à l'entrée (années révolues)
data['age_entree'] = ((data['date_entree'] - data['date_naissance']).dt.days / 365.25).astype(int)
data['age_sortie'] = ((data['date_sortie'] - data['date_naissance']).dt.days / 365.25).astype(int)
data['exposition_annees'] = (data['date_sortie'] - data['date_entree']).dt.days / 365.25

# Agrégation par âge
results = []
for age in range(20, 91):
    mask = (data['age_entree'] <= age) & (data['age_sortie'] >= age)
    subset = data[mask]
    E_x = subset['exposition_annees'].clip(upper=1).sum()
    D_x = ((subset['age_entree'] <= age) & (subset['age_sortie'] == age) & (subset['cause_sortie'] == 'deces')).sum()
    mu_x = D_x / E_x if E_x > 0 else np.nan
    q_x = 1 - np.exp(-mu_x) if not np.isnan(mu_x) else np.nan
    results.append({'age': age, 'E_x': E_x, 'D_x': D_x, 'mu_x': mu_x, 'q_x_brut': q_x})

table = pd.DataFrame(results)
print(table.head(10).to_string(index=False))

## Étape 4 — Lissage Whittaker-Henderson

Le lissage Whittaker-Henderson minimise un critère qui équilibre :
- **Fidélité** aux taux bruts (pondérée par l'exposition E_x)
- **Régularité** de la courbe (différences d'ordre 2)

Le paramètre `lambda_wh` contrôle l'arbitrage :
- Valeur faible → courbe proche des taux bruts (moins lisse)
- Valeur élevée → courbe très lisse (peut s'écarter des données)

**Valeur recommandée** : `lambda_wh = 100` pour des données actuarielles standard.

In [ ]:
# Paramètre de lissage — à ajuster si nécessaire
lambda_wh = 100

def whittaker_henderson(y, w, lam, d=2):
    """Lissage Whittaker-Henderson d'ordre d."""
    n = len(y)
    W = np.diag(w)
    # Matrice de différences d'ordre d
    D = np.diff(np.eye(n), n=d, axis=0)
    A = W + lam * D.T @ D
    return np.linalg.solve(A, W @ y)

# Appliquer sur les âges sans NaN
valid = table['q_x_brut'].notna() & (table['E_x'] > 0)
y = np.log(table.loc[valid, 'q_x_brut'].values.clip(1e-6))
w = table.loc[valid, 'E_x'].values

y_smooth = whittaker_henderson(y, w, lambda_wh)
table.loc[valid, 'q_x_lisse'] = np.exp(y_smooth)

print('Lissage appliqué.')
print(table[['age','q_x_brut','q_x_lisse']].dropna().head(10).to_string(index=False))

## Étape 5 — Visualisation et contrôle graphique

Comparer visuellement :
- Les taux bruts (points)
- La courbe lissée

Vérifier que la courbe lissée est **monotone croissante** après 40 ans et ne présente pas d'anomalie.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
t = table.dropna(subset=['q_x_brut'])
ax.scatter(t['age'], t['q_x_brut'], s=20, alpha=0.5, label='Taux bruts', color='steelblue')
ax.plot(t['age'], t['q_x_lisse'], color='crimson', linewidth=2, label=f'Lissage WH (λ={lambda_wh})')
ax.set_yscale('log')
ax.set_xlabel('Âge')
ax.set_ylabel('qx (échelle log)')
ax.set_title('Table de mortalité — taux bruts vs lissés')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Étape 6 — Calcul du SMR (Standardised Mortality Ratio)

Le SMR compare la mortalité observée dans le portefeuille à celle attendue selon une table de référence :

**SMR = Décès observés / Décès attendus**

- SMR < 1 : portefeuille moins mortel que la référence (effet de sélection)
- SMR > 1 : portefeuille plus mortel que la référence

Ici on utilise une table de référence simplifiée (Makeham). En production, remplacer par les qx TH/TF 00-02 réels.

In [ ]:
# Table de référence simplifiée (Makeham — à remplacer par TH/TF 00-02 en production)
ages_ref = np.arange(20, 91)
q_ref = pd.Series(
    0.0001 * np.exp(0.085 * (ages_ref - 20)),
    index=ages_ref
).clip(upper=0.5)

table['q_x_ref'] = table['age'].map(q_ref)
table['deces_attendus'] = table['E_x'] * table['q_x_ref']

D_obs = table['D_x'].sum()
D_att = table['deces_attendus'].sum()
SMR = D_obs / D_att

print(f'Décès observés  : {int(D_obs)}')
print(f'Décès attendus  : {D_att:.1f}')
print(f'SMR             : {SMR:.3f}')
if SMR < 0.9:
    print('=> Portefeuille significativement moins mortel que la référence')
elif SMR > 1.1:
    print('=> Portefeuille significativement plus mortel que la référence')
else:
    print('=> Mortalité proche de la référence')

## Étape 7 — Export de la table finale

La table finale contient pour chaque âge :
- `q_x_brut` : taux de mortalité brut observé
- `q_x_lisse` : taux lissé (à utiliser pour les calculs actuariels)
- `q_x_ref` : taux de la table de référence
- `SMR_local` : ratio observé/attendu par âge

Le fichier est exporté en CSV.

In [ ]:
table['SMR_local'] = (table['D_x'] / table['deces_attendus']).round(3)

output_cols = ['age','E_x','D_x','q_x_brut','q_x_lisse','q_x_ref','SMR_local']
table_finale = table[output_cols].round(6)

output_path = 'table_mortalite_experience.csv'
table_finale.to_csv(output_path, index=False)
print(f'Table exportée : {output_path}')
print(f'Âges couverts  : {table_finale["age"].min()} — {table_finale["age"].max()}')
print(f'SMR global     : {SMR:.3f}')
print(table_finale.tail(10).to_string(index=False))